# RAG-IDEArq — Retrieval: Híbrida + Multiquery

Notebook **nuevo** (no modifica los preservados). Compara estrategias de recuperación
sobre el mismo dataset, mismo LLM generador y mismas métricas RAGAS.

**Arregla además el bug de inyección de contexto**: el prompt se normaliza
(`{{var}}` → `{var}`) y se compila con `ChatPromptTemplate`, pasando mensajes a `ChatOllama`.
Funciona con prompts de Langfuse en sintaxis mustache (doble llave) o f-string (una llave).

**Estrategias**: `vector` (alpha=1.0 puro) · `hybrid_0.3/0.5/0.7` · `mq_vector` · `mq_hybrid_0.5`
(`mq_*` = multiquery + fusión RRF).

**Ejecutar en el nodo con servicios**: Weaviate (:8080), Ollama (:11434), Langfuse (:4000).


In [1]:
# ── Celda 1 · Configuración ──────────────────────────────────────────────────
from __future__ import annotations
import os, sys, re, json, time, warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Tuple

warnings.filterwarnings("ignore")

# PROJECT_ROOT = carpeta idearq-rag (padre de notebooks/)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
for cand in [PROJECT_ROOT / ".env", PROJECT_ROOT.parent / ".env"]:
    if cand.exists():
        load_dotenv(cand, override=True)
        print("Loaded env:", cand)
        break

# ── Palancas de retrieval a evaluar ──────────────────────────────────────────
EMBEDDING_KEY = "all-MiniLM-L6-v2"     # -> IdearqMiniLM_800_50_v2 (src.config.collection_name)
from src.config import collection_name
ROUTE_COLLECTION = collection_name(EMBEDDING_KEY)   # Route B and the notebook share this collection
GEN_LLM       = "Qwen3-4B-Instruct-2507"    # mejor combo conocido
GEN_PROMPT    = "one_shot"
GEN_TEMP      = 0.5
K             = 8                          # docs finales al contexto
RERANK        = False                       # reranking OFF (fase 2: contraproducente)
ALPHAS        = [0.3, 0.5, 0.7]
N_QUERIES     = 4                           # variantes multiquery
RRF_K         = 60
MQ_GENERATOR_MODEL = "hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q8_0"   # local Ollama (Qwen3-4B)
MQ_GENERATOR_TEMP  = 0.2
RAGAS_JUDGE   = "hf.co/unsloth/Qwen3-14B-GGUF:Q6_K"   # local Ollama judge (Qwen3-14B Q6_K)

# Phase G (context-only): append biblio RDF docs as extra retrieval context.
# Graph is NOT a strategy — never add graph_only to STRATEGIES.
GRAPH_CONTEXT = True           # Phase G: smoke CON grafo (decisión 2026-09-29)
GRAPH_PATH = PROJECT_ROOT / "data" / "biblio-graph" / "idearq-graph-instances-dedup-v2.nt"
GRAPH_TOP_N = 5
# Ollama base URL: resolve the LIVE services node from services.env (written by
# start_rag.sh). Do not hardcode 127.0.0.1 or a previous node — Slurm moves them.
def _load_ollama_base() -> str:
    candidates = [Path("/mnt/lustre/scratch/nlsas//home/csic/usi/eag/idearq-rag/services.env")]
    _lustre = os.environ.get("LUSTRE")
    if _lustre:
        candidates.append(Path(_lustre) / "idearq-rag" / "services.env")
    candidates.append(PROJECT_ROOT / "services.env")
    seen: set[str] = set()
    for p in candidates:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        if not p.is_file():
            continue
        try:
            lines = p.read_text().splitlines()
        except OSError as exc:
            print(f"  warn: cannot read {p}: {exc}")
            continue
        vals: dict[str, str] = {}
        for line in lines:
            line = line.strip()
            if line.startswith("export "):
                line = line[len("export "):]
            if "=" not in line or line.startswith("#"):
                continue
            k, v = line.split("=", 1)
            vals[k.strip()] = v.strip().strip('"').strip("'")
        raw = vals.get("OLLAMA_URL") or vals.get("OLLAMA_HOST")
        if raw:
            if "://" not in raw:
                raw = f"http://{raw}"
            return raw.rstrip("/").removesuffix("/v1").rstrip("/")
    env = (os.environ.get("OLLAMA_URL") or os.environ.get("OLLAMA_HOST") or "").strip()
    if env:
        print("  warn: services.env missing OLLAMA_*; using process env (may be stale)")
        if "://" not in env:
            env = f"http://{env}"
        return env.rstrip("/").removesuffix("/v1").rstrip("/")
    raise RuntimeError("Cannot resolve OLLAMA_BASE: no OLLAMA_URL/OLLAMA_HOST in services.env or env")

OLLAMA_BASE = _load_ollama_base()
print("Ollama base  :", OLLAMA_BASE)
DATASET_NAME  = "RAG-IDEArq-eval-v3-SIMPLE" # o "RAG-IDEArq-eval-v3-COMPLEX"
BM25_PROPS    = ["content", "tipologia_crono", "unidad_territorial"]

STRATEGIES = ["vector"] + [f"hybrid_{a}" for a in ALPHAS] + ["mq_vector", "mq_hybrid_0.5"]

SMOKE_ONLY = True    # True = pocas preguntas para validar; False = grid completo
SMOKE_N    = 3
if SMOKE_ONLY:  # smoke rápido (decisión 2026-09-29): 2 estrategias representativas
    STRATEGIES = ["vector"]  # viz check: A/B contra las micro-pruebas (vector vs mq_hybrid)

RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("Collection   :", EMBEDDING_KEY)
print("Strategies   :", STRATEGIES)
print("SMOKE_ONLY   :", SMOKE_ONLY)


Loaded env: /mnt/netapp1/Store_CSIC/home/csic/usi/eag/idearq-rag/.env
Ollama base  : http://c201-1:11434
PROJECT_ROOT : /mnt/netapp1/Store_CSIC/home/csic/usi/eag/idearq-rag
Collection   : gte-multilingual-base
Strategies   : ['vector', 'hybrid_0.3', 'hybrid_0.5', 'hybrid_0.7', 'mq_vector', 'mq_hybrid_0.5']
SMOKE_ONLY   : True


## 2 · Cliente Weaviate y modelos

In [2]:
from src.weaviate_conn import connect_weaviate
# ── Celda 2 · Weaviate + embeddings + LLMs ───────────────────────────────────
import weaviate
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

from src.config import EMBEDDINGS, LLMS, collection_name

w_client = connect_weaviate()
print("Weaviate ready:", w_client.is_ready())
COLLECTION = collection_name(EMBEDDING_KEY)
print("Collection:", COLLECTION, "| exists:", w_client.collections.exists(COLLECTION))
col = w_client.collections.get(COLLECTION)


class E5InstructEmbeddings(Embeddings):
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)
    def embed_documents(self, texts):
        return self.model.encode([f"passage: {t}" for t in texts], device=self.device).tolist()
    def embed_query(self, text):
        return self.model.encode([f"query: {text}"], device=self.device)[0].tolist()


def load_embeddings(key: str) -> Embeddings:
    cfg = EMBEDDINGS[key]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if cfg.get("model_class") == "E5InstructEmbeddings":
        return E5InstructEmbeddings(model_name=cfg["model_name"], device=device)
    mk = {"device": device}
    if cfg.get("trust_remote_code"):
        mk["trust_remote_code"] = True
    return HuggingFaceEmbeddings(model_name=cfg["model_name"], model_kwargs=mk,
                                 encode_kwargs={"device": device})


embedding = load_embeddings(EMBEDDING_KEY)
gen_llm = ChatOllama(model=LLMS[GEN_LLM], temperature=GEN_TEMP, num_ctx=16384, num_predict=-1, base_url=OLLAMA_BASE)
mq_llm  = ChatOllama(model=MQ_GENERATOR_MODEL, temperature=MQ_GENERATOR_TEMP, base_url=OLLAMA_BASE)
print("Embedding:", EMBEDDING_KEY, "| Gen LLM:", LLMS[GEN_LLM], "| MQ gen:", MQ_GENERATOR_MODEL)


Weaviate ready: True
Collection: IdearqGTE_800_50_v2 | exists: True


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Embedding: gte-multilingual-base | Gen LLM: hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q8_0 | MQ gen: hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q8_0


## 3 · Prompt (robusto a doble/una llave)

In [3]:
# ── Celda 3 · Carga del prompt (local-first, src.langfuse_monitor) ─────────
from src.langfuse_monitor import get_prompt

PROMPT_NAMES = {"zero_shot": "prompt_zero_shot", "one_shot": "prompt_one_shot", "few_shot": "prompt_few_shot"}

PROMPT = get_prompt(GEN_PROMPT)
print("Prompt cargado:", PROMPT_NAMES.get(GEN_PROMPT, GEN_PROMPT))
print("Variables del prompt:", PROMPT.input_variables)
assert "context" in PROMPT.input_variables and "question" in PROMPT.input_variables, \
    f"El prompt no expone context/question: {PROMPT.input_variables}"


Prompt cargado: prompt_one_shot
Variables del prompt: ['context', 'question']


## 4 · Dataset

In [4]:
# ── Celda 4 · Dataset (Langfuse, con fallback local) ─────────────────────────

def load_items():
    try:
        ds = _lf.get_dataset(name=DATASET_NAME)
        items = []
        for it in ds.items:
            q = it.input if isinstance(it.input, str) else (it.input or {}).get("question")
            gt = it.expected_output if isinstance(it.expected_output, str) else str(it.expected_output)
            meta = it.input if isinstance(it.input, dict) else {}
            if q:
                items.append({"question": q, "ground_truth": gt,
                              "tipo": meta.get("tipo"), "idioma": meta.get("idioma")})
        if items:
            print(f"Dataset Langfuse '{DATASET_NAME}': {len(items)} items")
            return items
    except Exception as e:
        print("Langfuse dataset no disponible ->", e)

    from data.eval_questions import EVAL_ITEMS
    items = [{"question": x[1], "ground_truth": x[2], "tipo": x[4], "idioma": x[5]} for x in EVAL_ITEMS]
    print(f"Dataset local eval_questions.py: {len(items)} items")
    return items


ITEMS = load_items()

# ── Selection for the viz check (2026-10-06) ────────────────────────────────
# ALL 32 gold labels now carry a `question`, so the old "geo + 3 biblio"
# selector would pick 32 questions and blow past viz's 8 h limit. Pin the
# smoke to the SAME eight questions as job 10180170 so the CSVs are directly
# comparable. Matched by distinctive substring, never by numeric index
# (load_items() may serve the Langfuse dataset in another order).
VIZ_SMOKE_PATTERNS = [
    "In what year did the Siret brothers excavate",
    "Which Chalcolithic site is the furthest from the city of Jaén",
    "Emergency excavations carried out by the Regional Government of Andalusia",  # #3 (reescrita por el usuario; pin actualizado 2026-10-08)
    "¿Es correcta la datación -1377±23 para Valencina",
    "¿Estas dataciones son compatibles con el periodo del Neolítico",
    "¿Cuál es la fecha más antigua para un yacimiento funerario megalítico",
    "Quines són les cronologies de les manifestacions funeràries del Mesolític",
    "Dataciones más antiguas (i.e, más altas) de yacimientos paleolíticos",
]

_picked: Dict[str, list] = {}
for _it in ITEMS:
    for _p in VIZ_SMOKE_PATTERNS:
        if _p in _it["question"]:
            _picked.setdefault(_p, []).append(_it)
_missing = [p for p in VIZ_SMOKE_PATTERNS if len(_picked.get(p, [])) != 1]
ITEMS = [_picked[p][0] for p in VIZ_SMOKE_PATTERNS if p in _picked and len(_picked[p]) == 1]
print(f"VIZ CHECK: {len(ITEMS)}/8 preguntas fijadas (mismo set que el job 10180170)")
for _i, _it in enumerate(ITEMS):
    print(f"   Q{_i+1}: {_it['question'][:72]}")
for _p in _missing:
    print("   MISSING/DUP:", _p)
if len(ITEMS) != 8:
    raise RuntimeError(f"smoke pin incompleto: {len(ITEMS)}/8 — revisar VIZ_SMOKE_PATTERNS")


Langfuse dataset no disponible -> name '_lf' is not defined
Dataset local eval_questions.py: 32 items
Preguntas a evaluar: 3


## 5 · Retrieval explícito (cliente Weaviate v4)

In [5]:
# ── Celda 5 · Retrieval: vector puro y híbrido ───────────────────────────────
RETURN_PROPS = ["content", "filename", "doc_type", "yacimiento_nombre",
                "tipologia_crono", "unidad_territorial", "year", "language", "chunk_index",
                "lat", "lon"]

def _to_docs(objects) -> List[Document]:
    out = []
    for o in objects:
        p = dict(o.properties)
        content = p.pop("content", "") or ""
        md = {k: v for k, v in p.items() if v is not None}
        md["uuid"] = str(o.uuid)
        md["score"] = getattr(o.metadata, "score", None)
        out.append(Document(page_content=content, metadata=md))
    return out

def vector_search(query: str, k: int = K) -> List[Document]:
    emb = embedding.embed_query(query)
    res = col.query.near_vector(near_vector=emb, limit=k,
                                return_metadata=["score"], return_properties=RETURN_PROPS)
    return _to_docs(res.objects)

def hybrid_search(query: str, alpha: float, k: int = K) -> List[Document]:
    emb = embedding.embed_query(query)
    res = col.query.hybrid(query=query, vector=emb, alpha=alpha, limit=k,
                           query_properties=BM25_PROPS,
                           return_metadata=["score"], return_properties=RETURN_PROPS)
    return _to_docs(res.objects)

print("Retrieval definido. Ejemplo:")
_demo = vector_search(ITEMS[0]["question"], k=3)
for d in _demo:
    print("  -", d.metadata.get("filename", "N/A"), "|", d.page_content[:60].replace("\n", " "))


Retrieval definido. Ejemplo:
  - 917_LaBastida_2015.pdf | 108.		Es probable que el hacha de talón y anillas procedente
  - 917_LaBastida_2015.pdf | 75.	 	Es posible que Siret sólo acudiese a Totana unos cuant
  - 917_LaBastida_2015.pdf | 230_ PRIMERAS INVESTIGACIONES EN LA BASTIDA (1869-2005) 556.


## 6 · Multiquery + fusión RRF

In [6]:
# ── Celda 6 · Multiquery y Reciprocal Rank Fusion ────────────────────────────
MQ_PROMPT = """Genera {n} reformulaciones distintas de la siguiente pregunta de investigación arqueológica.
Reglas:
- Conserva la intención y los nombres propios (yacimientos, autores, regiones, dataciones).
- Varía el vocabulario y el enfoque; incluye al menos UNA versión en español (el corpus documental está en español) y al menos una en inglés.
- Devuelve SOLO las preguntas, una por línea, sin numeración ni texto adicional.

Pregunta: {question}
"""

def generate_query_variants(question: str, n: int = N_QUERIES) -> List[str]:
    try:
        out = mq_llm.invoke(MQ_PROMPT.format(n=n, question=question))
        text = out.content if hasattr(out, "content") else str(out)
        lines = [re.sub(r"^\s*\d+[\.\)\-:\s]+", "", ln).strip()
                 for ln in text.splitlines() if ln.strip()]
        variants = [ln for ln in lines if len(ln) > 10][:n]
        return variants or [question]
    except Exception as e:
        print("  MQ gen error:", e)
        return [question]

def rrf_fuse(lists_of_docs: List[List[Document]], k: int = RRF_K, top_k: int = K) -> List[Document]:
    scores: Dict[Any, float] = {}
    docmap: Dict[Any, Document] = {}
    for docs in lists_of_docs:
        for rank, d in enumerate(docs, start=1):
            uid = d.metadata.get("uuid") or hash(d.page_content)
            docmap[uid] = d
            scores[uid] = scores.get(uid, 0.0) + 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]
    return [docmap[uid] for uid, _ in ranked]

print("Multiquery + RRF definidos. Ejemplo de variantes:")
for v in generate_query_variants(ITEMS[0]["question"]):
    print("  -", v[:100])


Multiquery + RRF definidos. Ejemplo de variantes:
  - When did the Siret brothers conduct excavations at the La Bastida de Totana site?
  - What year marked the excavation of La Bastida de Totana by the Siret brothers?
  - Which year corresponds to the Siret brothers' excavation of the La Bastida de Totana site?
  - In what year were the Siret brothers active at the La Bastida de Totana archaeological site?


## 7 · Estrategia → retrieval, y generación

In [7]:
# ── Celda 7 · Dispatcher de estrategias + run_rag ────────────────────────────
def retrieve(strategy: str, question: str) -> List[Document]:
    if strategy == "vector":
        return vector_search(question, K)
    if strategy.startswith("hybrid_"):
        return hybrid_search(question, float(strategy.split("_")[1]), K)
    if strategy in ("mq_vector", "mq_hybrid_0.5"):
        variants = [question] + generate_query_variants(question)
        if strategy == "mq_vector":
            lists = [vector_search(v, K) for v in variants]
        else:
            lists = [hybrid_search(v, 0.5, K) for v in variants]
        return rrf_fuse(lists)
    raise ValueError(f"Estrategia desconocida: {strategy}")

# Single source of truth for context serialisation (coordinates AND _distance_km).
# The old local copy here duplicated src.geo_route.ctx_text and misled a diagnosis
# (it never emitted the distance line). Never re-implement it in the notebook.
from src.geo_route import ctx_text as _ctx_text

# The filter route lives in a later cell; bind it lazily so this cell keeps working
# on its own (a NameError here would take the whole notebook down).
plan_geo_query = None
retrieve_filtered = None

def run_rag(strategy: str, question: str):
    t0 = time.time()
    # Route B (GeoJSON superlative / date questions): the deterministic planner
    # asks the collection a filtered question instead of a top-k similarity one.
    route = "topk"
    hit = None
    if plan_geo_query is not None:
        try:
            hit = retrieve_filtered(question, collection=ROUTE_COLLECTION)
        except Exception as _e:
            print("    filter route unavailable:", str(_e)[:90])

    # Once Route B fires it owns the question (empty-filter fix, 2026-10-05). A
    # filtered query that matched 0 objects used to fall through to top-k while the
    # CSV kept the label "filter:<kind>", so the row claimed a hit the model never
    # had. Now the route gains "VACIO, sin fallback topk" and the context is the
    # warning NOTE: unrelated similarity PDFs are never sold as filter results.
    if hit is None:
        docs = retrieve(strategy, question)
        t_ret = time.time() - t0
        context = "\n\n".join(_ctx_text(d) for d in docs[:K])
    else:
        # RouteResult: diagnostics are read as attributes, never printed by the lib.
        route = "filter:" + hit.plan["kind"]
        docs = hit.docs
        if hit.empty:
            route += " → VACÍO, sin fallback topk"
            docs = []
            print("    " + route)
        for _w in hit.warnings:
            print("    " + _w)
        for _e in hit.errors:
            print("    route error:", _e)
        print(f"    supporting PDFs: {hit.n_pdf_docs}")
        t_ret = time.time() - t0
        context = hit.context or "\n\n".join(_ctx_text(d) for d in docs[:K])
    if GRAPH_CONTEXT and not (hit and hit.empty):  # Phase G context-only: append graph docs (cell 8b helpers)
        try:
            _, _gdocs = maybe_append_graph(docs, question)
            if _gdocs:
                if hit is not None:
                    # Route B owns the context (empty-filter fix): append the graph
                    # docs to it instead of rebuilding, which would drop the filter
                    # note and the supporting PDFs.
                    context = context + "\n\n" + "\n\n".join(d.page_content for d in _gdocs)
                else:
                    context = docs_to_context_append(docs, _gdocs)
        except NameError:
            pass  # helpers de la celda 8b aún no definidos → sin grafo
    messages = PROMPT.format_messages(context=context, question=question)
    t1 = time.time()
    out = gen_llm.invoke(messages)
    answer = out.content if hasattr(out, "content") else str(out)
    t_gen = time.time() - t1
    return answer, docs, context, t_ret, t_gen, route

print("run_rag listo.")


run_rag listo.


# ── Route B: filter + aggregate for GeoJSON superlative questions ─────────
#
# Top-k by similarity cannot answer "which site is furthest from X": it returns the
# sites NEAREST X, so the correct answer never enters the context (Eras del Alcazar
# is absent from every top-100 at any alpha). These questions need the whole
# candidate set filtered on structured attributes, then an aggregate.
#
# The query plan is built DETERMINISTICALLY (no LLM): superlative words and period
# names are matched against the values that actually exist in the collection. That
# keeps it auditable and lets us measure it with hit@k against the gold labels.
#
# Context is always built through _ctx_text(), so coordinates reach the prompt.
# Changing the Weaviate collection is a parameter: validated on
# IdearqMiniLM_800_50_v2 (enriched) before the eval collection is backfilled.

In [ ]:
# ── Celda 15 · Route B (adaptador fino sobre src.geo_route) ──────────────────
# Toda la lógica de recuperación vive ahora en src.geo_route (importable y
# testeable). Esta celda solo la conecta: registra el embedder del notebook y
# expone los nombres que usan las demás celdas y los harnesses que la ejecutan.
# Aquí no se duplica ninguna lógica.
from src.geo_route import (
    GEO_WEAVIATE_URL,        # endpoint resuelto por el módulo (WEAVIATE_URL/services.env)
    GEO_FILTER_COLLECTION,   # constante legada, se mantiene por compatibilidad
    plan_geo_query,
    retrieve_filtered,
    as_legacy_dict,
    ctx_text,
    set_embedding,
)

# Compatibilidad: las celdas 8b/10 y los harnesses exec() usan el nombre antiguo.
_ctx_text = ctx_text

# El embedder lo construye el notebook (celda 2); se lo pasamos a la pierna
# vectorial de PDFs. Si aún no existe, el módulo degrada de forma visible.
if "embedding" in globals():
    set_embedding(embedding)

print("Route B listo (src.geo_route): plan_geo_query, retrieve_filtered, ctx_text")


## 8 · Verificación del prompt (contexto SÍ llega)

Esta celda imprime el prompt final. **Comprueba que aparece texto de los documentos.**
Si aquí no hay contexto, no tiene sentido evaluar nada más.

In [8]:
# ── Celda 8 · Smoke test de inyección de contexto ────────────────────────────
q0 = ITEMS[0]["question"]
_docs = retrieve("hybrid_0.5", q0)
_ctx = "\n\n".join(d.page_content for d in _docs[:K])
_msgs = PROMPT.format_messages(context=_ctx, question=q0)

print("Pregunta:", q0[:100])
print("Docs recuperados:", len(_docs), "| chars de contexto:", len(_ctx))
print("=" * 70)
for m in _msgs:
    print(f"[{m.type}] {str(m.content)[:600]}")
print("=" * 70)
assert len(_ctx) > 200, "Contexto vacío/corto: revisa retrieval o text_key"
ok = any(d.page_content[:40] in _ctx for d in _docs if d.page_content)
print("El contexto contiene texto de los documentos:", ok)

answer, docs, ctx, tr, tg, _route = run_rag("hybrid_0.5", q0)
print(f"\nLatencia retrieval={tr:.2f}s gen={tg:.2f}s")
print("Respuesta:", answer[:400])


Pregunta: In what year did the Siret brothers excavate the La Bastida de Totana site?
Docs recuperados: 10 | chars de contexto: 17444
[human] Eres un asistente experto en arqueología, historia y sistemas de información geográfica. Tu misión es responder preguntas técnicas usando los contextos proporcionados, que pueden estar en español, inglés, francés, catalán o portugués.

Sintetiza información de todos los contextos relevantes independientemente de su idioma y responde siempre en el mismo idioma en el que se realiza la pregunta.

REGLAS DE RAZONAMIENTO TÉCNICO:

    EVIDENCIA BIBLIOGRÁFICA Y CRONOLÓGICA:

        Cuando sea posible, cita explícitamente los puntos clave del contexto: autores, años, títulos de publicaciones, yacimi
El contexto contiene texto de los documentos: True

Latencia retrieval=0.11s gen=5177.35s
Respuesta: Los hermanos Siret realizaron las primeras excavaciones en La Bastida de Totana en **1886**, según se indica en el contexto proporcionado. Este dato se deri

## 8b · Grafo bibliográfico (solo contexto) + smoke

Context-only Phase G: biblio RDF docs may be appended to the retrieval context.
There is **no** `graph_only` strategy — `STRATEGIES` stays retrieval-only.
The full Phase G grid lives in `scripts/run_fase.py --fase G`; this cell only
validates graph load + SPARQL query + context merge (smoke).


In [ ]:
# ── Celda 8b · Phase G helpers — graph context only (no graph_only strategy) ──
# Existing retrieve()/run_rag() are left untouched. Use maybe_append_graph()
# or retrieve_with_graph() only when GRAPH_CONTEXT is True. The full Phase G
# grid (baseline {vector|hybrid_0.5} x graph {off,on}) is implemented by
# scripts/run_fase.py --fase G; the notebook smoke only validates
# load + SPARQL query + context merge.

GRAPH_AVAILABLE = False
_GRAPH = None

def _load_graph():
    """Load the biblio RDF graph once; never raises (prints skip instead)."""
    global _GRAPH, GRAPH_AVAILABLE
    if _GRAPH is not None:
        return _GRAPH
    try:
        import rdflib  # noqa: F401
    except ImportError:
        print("rdflib not installed — graph smoke skipped")
        return None
    if not GRAPH_PATH.is_file():
        print(f"Graph file missing — graph smoke skipped: {GRAPH_PATH}")
        return None
    from rdflib import Graph
    t0 = time.time()
    _GRAPH = Graph()
    _GRAPH.parse(GRAPH_PATH, format="nt")
    GRAPH_AVAILABLE = True
    print(f"Biblio graph loaded: {len(_GRAPH)} triples in {time.time() - t0:.1f}s")
    return _GRAPH

def sparql_biblio(question: str, top_n: int = GRAPH_TOP_N) -> List[Document]:
    """Keyword/title SPARQL match over the biblio graph (context-only docs)."""
    g = _load_graph()
    if g is None:
        return []
    keywords = [w for w in re.findall(r"[\wáéíóúñüÁÉÍÓÚÑÜ]+", question.lower()) if len(w) > 3][:8]
    keywords = [kw.replace('"', "").replace("\\", "") for kw in keywords]
    keywords = [kw for kw in keywords if kw]
    if not keywords:
        return []
    filters = " || ".join(f'CONTAINS(LCASE(STR(?title)), "{kw}")' for kw in keywords)
    sparql = (
        "PREFIX dcterms: <http://purl.org/dc/terms/> "
        "SELECT DISTINCT ?s ?title WHERE { ?s dcterms:title ?title . "
        f"FILTER({filters}) }} LIMIT 100"
    )
    try:
        rows = list(g.query(sparql))
    except Exception as exc:
        print("  sparql_biblio error:", exc)
        return []
    scored = []
    for row in rows:
        uri, title = str(row[0]), str(row[1])
        hits = sum(1 for kw in keywords if kw in title.lower())
        if hits:
            scored.append((hits, uri, title))
    scored.sort(key=lambda x: -x[0])
    docs = []
    for hits, uri, title in scored[:top_n]:
        docs.append(Document(
            page_content=f"[biblio_graph] {title} ({uri})",
            metadata={"source": "biblio_graph", "uri": uri, "title": title},
        ))
    return docs

def docs_to_context_append(docs: List[Document], graph_docs: List[Document]) -> str:
    """Append graph docs at the END of the retrieval context (context-only)."""
    parts = [_ctx_text(d) for d in docs[:K]]
    parts.extend(d.page_content for d in graph_docs)
    return "\n\n".join(parts)

def maybe_append_graph(docs: List[Document], question: str):
    """Return (docs, graph_docs). Graph docs only when GRAPH_CONTEXT is True."""
    if not GRAPH_CONTEXT:
        return docs, []
    return docs, sparql_biblio(question, GRAPH_TOP_N)

def retrieve_with_graph(strategy: str, question: str, graph_on=None):
    """retrieve() + optional graph docs appended at the end.

    Does not modify run_rag(); call it explicitly when GRAPH_CONTEXT is True.
    """
    docs = retrieve(strategy, question)
    on = GRAPH_CONTEXT if graph_on is None else graph_on
    gdocs = sparql_biblio(question, GRAPH_TOP_N) if on else []
    return docs, gdocs

def smoke_graph():
    """Validate load + SPARQL + context merge on ITEMS[0] (graph off vs on)."""
    if not ITEMS:
        print("smoke_graph: no ITEMS to run")
        return
    if _load_graph() is None:
        print("smoke_graph skipped (rdflib or graph file unavailable)")
        return
    q = ITEMS[0]["question"]
    t0 = time.time()
    docs = retrieve("hybrid_0.5", q)
    base_ctx = "\n\n".join(d.page_content for d in docs[:K])
    gdocs = sparql_biblio(q, GRAPH_TOP_N)
    graph_ctx = docs_to_context_append(docs, gdocs)
    ms = (time.time() - t0) * 1000
    print("question      :", q[:80])
    print(f"n_docs        : {len(docs)} retrieval + {len(gdocs)} graph")
    print(f"context_chars : off={len(base_ctx)} on={len(graph_ctx)} delta=+{len(graph_ctx) - len(base_ctx)}")
    print(f"ms (graph)    : {ms:.0f}")
    for d in gdocs[:3]:
        print("  graph hit   :", str(d.metadata.get("title", ""))[:80])

# Load once at cell run (guarded: prints skip and continues on failure).
_load_graph()
if SMOKE_ONLY:
    smoke_graph()
else:
    print("smoke_graph skipped (SMOKE_ONLY=False); full grid: scripts/run_fase.py --fase G")


## 9 · RAGAS

In [9]:
# ── Celda 9 · RAGAS con juez local ───────────────────────────────────────────
from ragas.llms.base import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerCorrectness
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import evaluate as ragas_evaluate
from ragas.dataset_schema import SingleTurnSample
from ragas import EvaluationDataset

class JSONCleaningLLM(LangchainLLMWrapper):
    def __call__(self, prompt, **kwargs):
        response = super().__call__(prompt, **kwargs)
        if isinstance(response, str):
            if response.startswith("```"):
                response = response.split("\n", 1)[1]
                if response.endswith("```"):
                    response = response.rsplit("\n", 1)[0]
            a, b = response.find("{"), response.rfind("}")
            if a >= 0 and b > a:
                response = response[a:b + 1]
        return response

raw_judge = ChatOllama(
    model=RAGAS_JUDGE, temperature=0.1, format="json", num_ctx=32768,
    think=False,  # Qwen3: no thinking for judge JSON,
    base_url=OLLAMA_BASE,  # same-node Ollama; ignore stale OLLAMA_HOST
)
ragas_llm = JSONCleaningLLM(raw_judge)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cpu"}, encode_kwargs={"device": "cpu"})
)
ragas_metrics = [
    Faithfulness(llm=ragas_llm),
]
# 2026-10-01 (2nd run): only faithfulness. Measured on the services node: judge prompt
# eval runs at 25 tok/s on CPU (~2 min for a 3k-token prompt) plus ~5 tok/s generation,
# so one judge call costs 8-13 min. Two metrics x 8 rows does not fit in viz's 8 h.
# answer_correctness was the one that timed out (Job[1]: TimeoutError at 3600 s).
METRIC_KEYS = ["faithfulness"]
# 2026-09-30: retrieval is graded deterministically with hit@k/MRR, so the two context
# metrics are dropped. ContextPrecision alone costs one judge call per retrieved
# context (8-10 per row) and caused every timeout NaN.
METRIC_KEYS = ["faithfulness", "answer_correctness"]
ragas_run_config = RunConfig(max_workers=1, max_retries=10, max_wait=120, timeout=3600,
                             log_tenacity=True)  # fix NaN: 900s no caben 10 llamadas de precision (2026-09-30)
print("RAGAS configurado. Juez:", RAGAS_JUDGE)


/tmp/ipykernel_767791/3453372089.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerCorrectness
/tmp/ipykernel_767791/3453372089.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerCorrectness
/tmp/ipykernel_767791/3453372089.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import Faithfulness, 

RAGAS configurado. Juez: hf.co/unsloth/Qwen3-14B-GGUF:Q6_K


/tmp/ipykernel_767791/3453372089.py:29: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(
/mnt/lustre/scratch/nlsas/home/csic/usi/eag/idearq-rag/env_rag/lib/python3.12/site-packages/ragas/utils.py:113: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  return self.new_target(*args, **kwargs)


## 10 · Ejecución de la ablación

In [10]:
# ── Celda 10 · Grid: estrategia × pregunta + RAGAS ───────────────────────────
METRIC_KEYS = ["faithfulness", "context_precision", "context_recall", "answer_correctness"]
all_results: List[Dict[str, Any]] = []

for strategy in STRATEGIES:
    print(f"\n{'='*70}\nESTRATEGIA: {strategy}\n{'='*70}")
    samples, indices = [], []
    for i, item in enumerate(ITEMS):
        q, gt = item["question"], item["ground_truth"]
        print(f"  Q{i+1}/{len(ITEMS)}: {q[:60]}...")
        try:
            answer, docs, ctx, tr, tg, route = run_rag(strategy, q)
            samples.append(SingleTurnSample(
                user_input=q, response=answer, reference=gt,
                retrieved_contexts=[_ctx_text(d)[:800] for d in docs[:K]]))
            idx = len(all_results)
            indices.append(idx)
            row = {"strategy": strategy, "question": q, "ground_truth": gt,
                   "tipo": item.get("tipo"), "idioma": item.get("idioma"),
                   "answer": answer, "n_docs": len(docs),
                   "latency_retrieval_s": tr, "latency_gen_s": tg,
                   "filenames": ";".join(d.metadata.get("filename", "N/A") for d in docs),
                   "context_chars": len(ctx),
                   "graph_context": GRAPH_CONTEXT,
                   "route": route,
                   "yacimiento_ids": ";".join(
                       str(d.metadata.get("yacimiento_id"))
                       for d in docs if d.metadata.get("yacimiento_id") is not None)}
            row.update({k: None for k in METRIC_KEYS})
            row.update({"prompt_tokens_gen": None, "completion_tokens_gen": None,
                        "prompt_tokens_judge": None, "completion_tokens_judge": None,
                        "total_tokens": None})
            all_results.append(row)
            print(f"    OK ({tr:.1f}s+{tg:.1f}s, {len(docs)} docs)")
        except Exception as e:
            print("    ERROR:", e)
            row = {"strategy": strategy, "question": q, "ground_truth": gt,
                   "tipo": item.get("tipo"), "idioma": item.get("idioma"),
                   "answer": f"Error: {e}", "n_docs": 0,
                   "latency_retrieval_s": 0, "latency_gen_s": 0,
                   "filenames": "", "context_chars": 0,
                   "graph_context": GRAPH_CONTEXT}
            row.update({k: None for k in METRIC_KEYS})
            row.update({"prompt_tokens_gen": None, "completion_tokens_gen": None,
                        "prompt_tokens_judge": None, "completion_tokens_judge": None,
                        "total_tokens": None})
            all_results.append(row)

    if samples:
        print(f"  RAGAS sobre {len(samples)} muestras...")
        try:
            res = ragas_evaluate(dataset=EvaluationDataset(samples=samples),
                                 metrics=ragas_metrics, llm=ragas_llm,
                                 embeddings=ragas_emb, run_config=ragas_run_config)
            for j, scores in enumerate(res.scores):
                for k, v in scores.items():
                    mk = str(k).lower()
                    if mk in all_results[indices[j]]:
                        try:
                            all_results[indices[j]][mk] = float(v)
                        except (TypeError, ValueError):
                            pass
            print("  Métricas Q1:", {k: round(float(v), 3) for k, v in res.scores[0].items()})
        except Exception as e:
            print("  RAGAS error:", e)



ESTRATEGIA: vector
  Q1/3: In what year did the Siret brothers excavate the La Bastida ...
    OK (12.2s+4511.0s, 10 docs)
  Q2/3: Which Chalcolithic site is the furthest from the city of Jaé...
    ERROR: peer closed connection without sending complete message body (incomplete chunked read)
  Q3/3: Emergency excavations carried out by the Regional Government...
    ERROR: Query call with protocol GRPC search failed with message The request to Weaviate failed after 5 retries. Details: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:10.120.201.1:50051: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:10.120.201.1:50051: Failed to connect to remote host: connect: Connection refused (111)"}"
>.
  RAGAS sobre 1 muest

/mnt/lustre/scratch/nlsas/home/csic/usi/eag/idearq-rag/env_rag/lib/python3.12/site-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/mnt/lustre/scratch/nlsas/home/csic/usi/eag/idearq-rag/env_rag/lib/python3.12/site-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(
/mnt/lustre/scratch/nlsas/home/csic/usi/eag/idearq-rag/env_rag/lib/python3.12/site-packages/ragas/evaluation.py:161: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Exception raised in Job[0]: ConnectError(All connection attempts failed)
Exception raised in Job[1]: ConnectError(All connection attempts failed)
Exception raised in Job[2]: ConnectError(All connection attempts failed)
Exception raised in Job[3]: ConnectError(All connection attempts failed)


  Métricas Q1: {'faithfulness': nan, 'context_precision': nan, 'context_recall': nan, 'answer_correctness': nan}

ESTRATEGIA: hybrid_0.3
  Q1/3: In what year did the Siret brothers excavate the La Bastida ...
    ERROR: Query call with protocol GRPC search failed with message The request to Weaviate failed after 5 retries. Details: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:10.120.201.1:50051: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:10.120.201.1:50051: Failed to connect to remote host: connect: Connection refused (111)", grpc_status:14}"
>.
  Q2/3: Which Chalcolithic site is the furthest from the city of Jaé...
    ERROR: Query call with protocol GRPC search failed with message The request to Weaviate failed after 5

## 11 · Guardar resultados

# ── Retrieval metrics: hit@k and MRR (deterministic, no LLM) ────────────
#
# These grade RETRIEVAL, not the answer: is the gold document in the returned set,
# and how high? Unlike the RAGAS context metrics they cost nothing and always give
# the same number, which is what makes them usable to compare runs.

## Inputs

- `data/eval_gold_labels.py` holds the documents each question must surface.
- Rows carry `route` (topk or filter:<kind>) and `yacimiento_ids` for the match.

In [ ]:
import unicodedata

from data.eval_gold_labels import GOLD_LABELS, gold_docs


def _norm_filename(name) -> str:
    """Accent/case-folded filename without the .pdf extension (tolerant match key)."""
    s = unicodedata.normalize("NFKD", str(name or ""))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace("\xad", "").strip().lower()
    return s[:-4] if s.endswith(".pdf") else s


def _gold_keys(question: str):
    """Gold identifiers: (yacimiento_id | name) pairs + filename-only keys for PDF labels."""
    idx = next((i for i, lb in GOLD_LABELS.items()
                if lb.get("question", "").strip() == (question or "").strip()), None)
    if idx is None:
        return None
    keys = set()
    for doc in gold_docs(idx):
        if doc.get("yacimiento_id") is not None:
            keys.add(("id", str(doc["yacimiento_id"])))
        if doc.get("yacimiento_nombre"):
            keys.add(("name", doc["yacimiento_nombre"].strip().lower()))
        # PDF gold docs carry only a filename (no yacimiento_id) -> match them by filename.
        if doc.get("filename") and doc.get("yacimiento_id") is None:
            keys.add(("file", _norm_filename(doc["filename"])))
    return keys or None


def annotate_retrieval_metrics(all_results, k: int = K) -> list:
    """Add hit_at_k and reciprocal_rank to every row (None when unlabelled)."""
    for row in all_results:
        row["hit_at_k"] = None
        row["reciprocal_rank"] = None
        row["labelled"] = False
        keys = _gold_keys(row.get("question", ""))
        if not keys:
            continue
        row["labelled"] = True
        found_ids = [x for x in str(row.get("yacimiento_ids", "")).split(";") if x]
        rank = None
        # 1) yacimiento-id path, unchanged: GeoJSON labels keep scoring exactly as before.
        for pos in range(k):
            if pos >= len(found_ids):
                break
            if ("id", found_ids[pos]) in keys:
                rank = pos + 1
                break
        # 2) filename path, used only when the label declares filename-only gold docs.
        if rank is None and any(kind == "file" for kind, _ in keys):
            gold_files = {val for kind, val in keys if kind == "file"}
            found_files = [_norm_filename(x)
                           for x in str(row.get("filenames", "")).split(";") if x]
            for pos in range(k):
                if pos >= len(found_files):
                    break
                if found_files[pos] in gold_files:
                    rank = pos + 1
                    break
        row["hit_at_k"] = 1.0 if rank else 0.0
        row["reciprocal_rank"] = round(1.0 / rank, 4) if rank else 0.0
    return all_results


print("metricas de retrieval definidas: hit_at_k, reciprocal_rank (MRR)")

In [ ]:
annotate_retrieval_metrics(all_results)
for _r in all_results:
    print(f"   route={str(_r['route']):26} hit@k={_r['hit_at_k']} rr={_r['reciprocal_rank']} | {_r['question'][:58]}")
_lab = [r for r in all_results if r["hit_at_k"] is not None]
print("hit@k medio (preguntas etiquetadas):",
      round(sum(r["hit_at_k"] for r in _lab) / max(1, len(_lab)), 3))


In [11]:
# ── Celda 11 · CSV de resultados ─────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(all_results)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
tag = "viz_geocheck"
out_path = RESULTS_DIR / f"eval_ablation_retrieval_{tag}_{ts}.csv"
df.to_csv(out_path, index=False)
print("Guardado:", out_path, f"({len(df)} filas)")


Guardado: /mnt/netapp1/Store_CSIC/home/csic/usi/eag/idearq-rag/data/results/eval_ablation_retrieval_smoke_20260923_155040.csv (18 filas)


## 12 · Análisis (Δ vs baseline + t-test)

In [12]:
# ── Celda 12 · Resumen por estrategia ────────────────────────────────────────
from scipy import stats

mcols = [c for c in METRIC_KEYS if df[c].notna().any()]
summary = df.groupby("strategy")[mcols].mean().round(4)
print("=== Media por estrategia ===\n", summary.to_string())

if "vector" in summary.index and len(ITEMS) > 1:
    print("\n=== Δ vs baseline 'vector' ===")
    print((summary - summary.loc["vector"]).round(4).to_string())

    print("\n=== t-test pareado vs 'vector' (por pregunta) ===")
    for strat in [s for s in STRATEGIES if s != "vector"]:
        for m in mcols:
            a = df[df.strategy == "vector"].sort_values("question")[m]
            b = df[df.strategy == strat].sort_values("question")[m]
            if len(a) == len(b) and len(a) > 1 and a.notna().all() and b.notna().all():
                t, p = stats.ttest_rel(a, b)
                print(f"  {strat:16s} {m:20s} p={p:.4f} {'*' if p < 0.05 else ''}")

if df["tipo"].notna().any():
    print("\n=== Por tipo ===")
    print(df.groupby(["strategy", "tipo"])[mcols].mean().round(3).to_string())
if df["idioma"].notna().any():
    print("\n=== Por idioma ===")
    print(df.groupby(["strategy", "idioma"])[mcols].mean().round(3).to_string())


=== Media por estrategia ===
 Empty DataFrame
Columns: []
Index: [hybrid_0.3, hybrid_0.5, hybrid_0.7, mq_hybrid_0.5, mq_vector, vector]

=== Δ vs baseline 'vector' ===
Empty DataFrame
Columns: []
Index: [hybrid_0.3, hybrid_0.5, hybrid_0.7, mq_hybrid_0.5, mq_vector, vector]

=== t-test pareado vs 'vector' (por pregunta) ===

=== Por tipo ===
Empty DataFrame
Columns: []
Index: [(hybrid_0.3, simple), (hybrid_0.5, simple), (hybrid_0.7, simple), (mq_hybrid_0.5, simple), (mq_vector, simple), (vector, simple)]

=== Por idioma ===
Empty DataFrame
Columns: []
Index: [(hybrid_0.3, en), (hybrid_0.5, en), (hybrid_0.7, en), (mq_hybrid_0.5, en), (mq_vector, en), (vector, en)]
